# Prepare the Spatial Metabolomics data (using zonist)

Our "spatially aware" pathways analysis, for a given **spatial metabolomics dataset**, requires:
1. Formatting the data to be appropriate for pathway scores computation
2. To obtain the matrix of pathways (or metabolite sets) activity scores across all pixels on the correctly formatted data, and
3. To perform the downstream analysis of such matrix of pathways scores

However, sub-steps needed for performing 1. are multiple, and are here described. Many steps are automatized by our lib, and we demonstrated it here!

_Note_ : For deeper information, check the "annex2" notebook in appendix folder.


In [1]:
import os
import numpy as np
import pandas as pd
import csv as csv

In [2]:
import zonist.data as zd

INFO:zonist.data:
Time: 20241028-12.42.38


## i ) Load annotations

_Note_: : allows files generated by Metaspace, and also manually curated and/or prepared files having the compulsory columns (mz, formula, moleculeIds, moleculeNames, database)


In [3]:
mydata_dir = "/home/johanna/spatial_thesis/spatial_metabolomics/data"
#"/mnt/cbib/thesis_gbm/spatial_thesis/spatial_metabolomics/data"
anno_dict = {'HMDB': "brain-ellis_hmdb_annotations.csv",
             'LipidMaps': "brain-ellis_lipidmaps_annotations.csv",
            'custom_kegg': "brain-ellis_kegg_annot-manually.csv"}  

In [4]:
me_annots = zd.MetabolomeAnnotations(name="demo", 
                            data_dir=mydata_dir, 
                            subfolder="brain-ellis",
                            annotation_files=anno_dict)

In [5]:
me_annots.load_annotation()

INFO:zonist.data:Loading annotations
INFO:zonist.data:  loading HMDB
INFO:zonist.data:  loading LipidMaps
INFO:zonist.data:  loading custom_kegg


In [6]:
me_annots.annotationtbl.head(1)

,mz,formula,moleculeIds,moleculeNames,database
0,551.503387,C35H66O4,HMDB0033604,Artemoin A,HMDB


## ii) Load reference mappings to KEGG identifiers, and perform no-kegg identifiers translation


*First*: Load the reference mappings of metabolites identifiers from "no-KEGG" to KEGG

In [7]:
ref_files_dict = {'HMDB': "HMDB-to-KEGG_05-07-24.tsv",
              'LipidMaps': "LipidMaps-to-KEGG_10-18-24.tsv"}

In [8]:
me_refmap = zd.MetaboliteReferenceMappings(
    data_dir=mydata_dir,
    subfolder="metabolites-mappings",
    mapping_files=ref_files_dict
)

In [9]:
me_refmap.load_mappings()

INFO:zonist.data:Loading reference files for mapping compounds identifiers
INFO:zonist.data:  loading HMDB
INFO:zonist.data: your loaded dataframe :
           HMDB    KEGG
0  HMDB0000001  C01152
1  HMDB0000002  C00986
INFO:zonist.data:  loading LipidMaps
INFO:zonist.data: your loaded dataframe :
       LipidMaps    KEGG
0  LMFA08030003  C11839
1  LMFA08030004  C11841


*Second*: convert the metabolome annotations using the loaded reference mappings

To recall, the `.annotationtbl` attribute of the annotations object contains the HMDB and LipidMaps identifiers (which must be convertd to KEGG ones).


Do the translation effortless with this simple line ! :


In [10]:
me_annots.convert_ids_to_kegg(me_refmap.table)

INFO:zonist.data:Translating ids in the metabolome annotation table, to KEGG


Note that the new column **KEGG** appears in the table:

In [11]:
me_annots.annotationtbl.sample(2)

,mz,formula,moleculeIds,moleculeNames,KEGG,database
1190,417.336322,C27H44O3,HMDB0030024,Sarsasapogenin,C03963,HMDB
476,861.790567,C55H104O6,HMDB0005361,TG(16:0/16:0/20:1(11Z)),C00422,HMDB


In [12]:
# the above added the column KEGG to the .annotationtbl, automatically
me_annots.annotationtbl.shape

(236, 6)

## iii) GMT 

In [13]:
me_gmt = zd.MetabolicGmt(
    data_dir=mydata_dir,
    subfolder="metabolites-mappings",
    official_file="KEGG_hsa_pathways_compounds_R110.gmt",
    custom_file="user-defined-pathways-or-sets.gmt")

In [14]:
me_gmt.load()

INFO:zonist.data: The first line of your custom gmt file:
  abbreviation: 'usr-lac';   name : 'Lactate metabolism (Lac,Pyr,acCoA,LacCoA)';  metabolites  (first three) : '['C00186', 'C00022', 'C00024']'.
INFO:zonist.data:success, attributes : gmt_piled, pathways_ids


**Important**: join with the annotated $mz$ in the annotations object (`.annotationtbl` attribute)

In [16]:
me_gmt.join_mz(me_annots.annotationtbl[["mz", "KEGG"]])

In [17]:
me_gmt.write_to_json(
    dataset_subfolder="brain-ellis", mz_or_kegg_choice="mz")

INFO:zonist.data:saved json file to : /home/johanna/spatial_thesis/spatial_metabolomics/data/brain-ellis/gmt_mz.json


## iv) Quantifications matrix

The quantification matrix must have unique $mz$ features that perfectly match with those in the annotations. Two alternative ways of obtaining this matrix:

1. from the imzML+ibd files: this requires careful formatting as the $mz$ features in this file are never identical to those in the annotations (because the annotations keep the theoretical 'm/z' of the databases used by Metaspace).
OR,
2. by downloading the matrix from Metaspace and formatting it.

The matrix treated here was obtained as specified in numeral 2. above.

_Note_:
The matrix in our example from Metaspace has doublets of $mz$ features, because several databases (LipidMaps and HMDB) yielded hits for a same row (thus duplicating the row). Moreover, there are supplementary columns that GSVA will not accept: _formula_, _moleculeNames_, _adduct_, etc.  To run GSVA it is compulsory to have unique $mz \cdot coords$. This means, in rows, the $mz$ features, and in columns the 'x,y' coordinates ($coords$).

In [18]:
pre_mx_df = pd.read_csv(
    os.path.join(mydata_dir, "brain-ellis", 
                 "brain-ellis-mx-subset.csv"), 
    sep=",", comment="#", header=0)

In [20]:
cols_keep = ['mz']
cols_keep += pre_mx_df.columns[5:].tolist()
mx_df = pre_mx_df[cols_keep]
del pre_mx_df

In [21]:
mx_df = mx_df.sort_values(by='mz', ascending=True)
mx_df = mx_df.drop_duplicates()
mx_df.shape

(310, 171151)

In [23]:
mx_df.T.to_csv(os.path.join(mydata_dir, "brain-ellis", 
                 "brain-ellis-mx-clean-transposed.tsv"), 
               quoting=csv.QUOTE_NONNUMERIC, 
               sep="\t", index=True, header=False)

<div class="alert alert-block alert-danger">
<b>ALERT! Too many columns:</b> Files with ~200K of columns are slow to open (or write) in Python, and tremendously slow to open (or write) in R (~30 minutes !!). Moreover, in spatial metabolomics the mz features will be always less than 10K. To save time when reading/writing files with such a huge amount of columns, the transposed version is saved as .tsv file.
</div>

The GSVA step is treated in next notebook

In the terminal, from your home or working directory, you launch the R script:
```
Rscript zonist/src/r/$THE_SCRIPT ~/spatial_thesis/spatial_metabolomics/data/brain-ellis brain-ellis-mx-clean-transposed.tsv gmt_mz.json
```
it requires 13 hours of computing time, despite parallelization.